<h1><strong><center> Final Model

После тюнинга гиперпараметров моделей с помощью Optuna, мы получили улучшение результатов всех моделей, кроме ExtraTreesClassifier. Поэтому в итог возьмем все базовые модели после Optuna, кроме ExtraTreesClassifier. Ее оставим с дефолтными гиперпараметрами.

In [36]:
import pandas as pd
import pickle
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.base import BaseEstimator, ClassifierMixin
from catboost import CatBoostClassifier
import warnings

warnings.filterwarnings('error')
warnings.filterwarnings('ignore')

In [37]:
file_path = '../../data/data_for_final_models/AgglomerativeClustering_generated_features.csv'
data = pd.read_csv(file_path)

X = data.drop(columns='Cluster')
y = data['Cluster']

In [38]:
# Загрузка моделей

model_path = '../../models/2_optuna_models/optuna_catboost_model.cb'
catboost_model = CatBoostClassifier()
catboost_model.load_model(model_path)

with open('../../models/2_optuna_models/optuna_gradient_boosting_model.pkl', 'rb') as file:
    gradient_boosting_model = pickle.load(file)

with open('../../models/2_optuna_models/optuna_logistic_regression_model.pkl', 'rb') as file:
    logistic_regression_model = pickle.load(file)

with open('../../models/2_optuna_models/optuna_random_forest_model.pkl', 'rb') as file:
    random_forest_model = pickle.load(file)

with open('../../models/1_default_models/et_model.pkl', 'rb') as file:
    extra_trees_model = pickle.load(file)

In [39]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

catboost_preds, lr_preds, rf_preds, et_preds, gb_preds, meta_labels = [], [], [], [], [], []

for train_idx, val_idx in kf.split(X):
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    catboost_preds.append(catboost_model.predict(X_val))
    lr_preds.append(logistic_regression_model.predict(X_val))
    rf_preds.append(random_forest_model.predict(X_val))
    et_preds.append(extra_trees_model.predict(X_val))
    gb_preds.append(gradient_boosting_model.predict(X_val))

    meta_labels.append(y_val)

In [40]:
X_meta = pd.DataFrame({
    'catboost': [item for sublist in catboost_preds for item in sublist],
    'lr': [item for sublist in lr_preds for item in sublist],
    'rf': [item for sublist in rf_preds for item in sublist],
    'et': [item for sublist in et_preds for item in sublist],
    'gb': [item for sublist in gb_preds for item in sublist]
})

y_meta = [item for sublist in meta_labels for item in sublist]

In [41]:
X_meta['catboost'] = X_meta['catboost'].apply(lambda x: x[0])

In [42]:
'''
Обертка для XGBClassifier, добавляющая метод __sklearn_tags__  
для совместимости со scikit-learn 1.7 и выше.  
Решает проблему с DeprecationWarning
'''
class SklearnXGBClassifier(xgb.XGBClassifier, BaseEstimator, ClassifierMixin):
    def __sklearn_tags__(self):
        return {
            'non_deterministic': True,
            'requires_fit': True,
            'X_types': ['2darray'],
        }

In [43]:
meta_model = xgb.XGBClassifier(random_state=42, tree_method='auto')
meta_model.fit(X_meta, y_meta);

In [44]:
catboost_final_preds = catboost_model.predict(X)
lr_final_preds = logistic_regression_model.predict(X)
rf_final_preds = random_forest_model.predict(X)
et_final_preds = extra_trees_model.predict(X)
gb_final_preds = gradient_boosting_model.predict(X)

X_meta_test = pd.DataFrame({
    'catboost': catboost_final_preds.ravel(),
    'lr': lr_final_preds,
    'rf': rf_final_preds,
    'et': et_final_preds,
    'gb': gb_final_preds
})

final_preds = meta_model.predict(X_meta_test)
accuracy = balanced_accuracy_score(y, final_preds)
print(f'Balanced Accuracy: {accuracy:.4f}')

Balanced Accuracy: 0.9997


Видим повышение точности по сравнению с блендинговых базовых моделей с дефолтными параметрами. Теперь сохраним модели и мета-модель.

In [45]:
with open('../../models/3_final_models/optuna_gradient_boosting_model.pkl', 'wb') as file:
    pickle.dump(gradient_boosting_model, file)

with open('../../models/3_final_models/optuna_logistic_regression_model.pkl', 'wb') as file:
    pickle.dump(logistic_regression_model, file)
    
with open('../../models/3_final_models/optuna_random_forest_model.pkl', 'wb') as file:
    pickle.dump(random_forest_model, file)
    
with open('../../models/3_final_models/et_model.pkl', 'wb') as file:
    pickle.dump(extra_trees_model, file)
    
with open('../../models/3_final_models/meta_model_xgboost.pkl', 'wb') as file:
    pickle.dump(meta_model, file)

catboost_model.save_model('../../models/3_final_models/catboost_model.cb')